# 00 — Label Feasibility & GapMind Aggregation

**Purpose**: Validate the four feasibility questions before writing `RESEARCH_PLAN.md`:

1. Where does cultured-vs-uncultured status live in BERDL?
2. How is `gapmind_pathways` structured (multi-row per (genome, pathway))?
3. After CheckM filtering, what is the class balance and within-family overlap?
4. Does the cultivability signal survive a CheckM-completeness control?

All queries are direct Spark SQL against `kbase_ke_pangenome` on-cluster.

## Headline findings (TL;DR)

- **Label source**: `gtdb_metadata.ncbi_genome_category`. `'none'` → isolate (211,887); MAG/env/single-cell → uncultured (81,172). ID format fracture between `gtdb_metadata.accession` (`RS_`/`GB_` prefixed) and `gapmind_pathways.genome_id` (bare); strip with `REGEXP_REPLACE`.
- **GapMind row uniqueness**: per `(genome_id, metabolic_category, pathway, sequence_scope, rule)`, with multiple rule rows per pathway. Canonical aggregation per the `[pgp_pangenome_ecology]` pitfall: `MAX(score_simplified) GROUP BY genome_id, pathway` — `score_simplified` is binary (0.0 / 1.0).
- **Phylogenetic coverage**: 1767 GTDB families; **203 families have ≥5 of both labels**, 144 have ≥10, 103 have ≥20 — enough for family-stratified train/test.
- **Signal survives CheckM control**: at CheckM ≥95% / contam ≤5%, MAG (n=27,581) carbon-pathway completeness is **40.0%** vs isolate (n=190,011) **67.9%** — a 28pp gap that is *not* an assembly-incompleteness artifact.

In [ ]:
from berdl_notebook_utils.setup_spark_session import get_spark_session
spark = get_spark_session()
print(f'Spark {spark.version}')

## Q1 — Label source: `ncbi_genome_category`

`gtdb_metadata` carries cultivation provenance in `ncbi_genome_category`:

| Value | n | Interpretation |
|---|---:|---|
| `none` | 211,887 | Isolate (cultured) |
| `derived from metagenome` | 75,931 | MAG (uncultured) |
| `derived from environmental sample` | 4,805 | Env-derived (uncultured) |
| `derived from single cell` | 436 | SAG (uncultured) |

`ncbi_isolate` is unreliable on its own — 73,896 MAGs carry an isolate-style ID (`MAG-XXX`). `ncbi_assembly_type` is uniformly `'na'` in this snapshot and not informative.

In [ ]:
spark.sql('''
SELECT ncbi_genome_category, COUNT(*) as n
FROM kbase_ke_pangenome.gtdb_metadata
GROUP BY ncbi_genome_category ORDER BY n DESC
''').show(truncate=False)

## Q2 — GapMind row uniqueness and the aggregation rule

`gapmind_pathways` is **305M rows × 292,806 genomes × 80 pathways**. Multiple rows per `(genome, pathway, sequence_scope)` correspond to different scoring rules within the pathway. Per the `[pgp_pangenome_ecology]` pitfall in `docs/pitfalls.md`, the canonical reduction is:

```sql
MAX(score_simplified) GROUP BY genome_id, pathway, metabolic_category, sequence_scope
```

`score_simplified` is binary (0.0 / 1.0), so `MAX` returns 1.0 iff *any* rule in the pathway is complete — the appropriate "can the organism do this" reduction.

Categories and scopes:

- `metabolic_category`: `aa` (amino-acid biosynthesis, 46M rows, ~17 pathways), `carbon` (carbon utilization, 259M rows, ~62 pathways). **No separate vitamin/cofactor category.**
- `sequence_scope`: `aux` / `all` / `core` — use `all` to include both species-core and species-auxiliary hits.
- `score_category`: 5-level coarsening of `score`.

In [ ]:
spark.sql('''
SELECT metabolic_category, sequence_scope, COUNT(*) as n_rows
FROM kbase_ke_pangenome.gapmind_pathways
GROUP BY 1, 2 ORDER BY 1, 2
''').show()

## Q3 — Class balance and within-family overlap after CheckM filter

Filter to MIMAG-HQ-ish quality (`checkm_completeness ≥ 90`, `checkm_contamination ≤ 5`) so that incomplete MAGs do not pollute downstream pathway-completeness comparisons.

| Class | All | After CheckM filter |
|---|---:|---:|
| Isolate | 211,887 | **209,698** |
| MAG | 75,931 | 41,240 |
| Env sample | 4,805 | 2,847 |
| Single cell | 436 | 77 |

Family overlap (HQ-filtered):

- 1767 GTDB families total
- 589 families have ≥1 isolate (the cultivation gap: only **33%** of bacterial families have any cultured representative)
- 1652 families have ≥1 MAG
- **203 families have ≥5 of both** → usable for family-stratified holdout
- 144 families have ≥10 of both, 103 families have ≥20 of both

In [ ]:
spark.sql('''
WITH labeled AS (
  SELECT
    g.genome_id,
    CASE WHEN m.ncbi_genome_category = 'none' THEN 'isolate'
         WHEN m.ncbi_genome_category IN ('derived from metagenome', 'derived from environmental sample', 'derived from single cell') THEN 'mag'
         ELSE 'unknown' END as label,
    t.family
  FROM kbase_ke_pangenome.genome g
  JOIN kbase_ke_pangenome.gtdb_metadata m ON g.genome_id = m.accession
  JOIN kbase_ke_pangenome.gtdb_taxonomy_r214v1 t ON g.genome_id = t.genome_id
  WHERE CAST(m.checkm_completeness AS DOUBLE) >= 90 AND CAST(m.checkm_contamination AS DOUBLE) <= 5
),
fam AS (
  SELECT family,
    SUM(CASE WHEN label='isolate' THEN 1 ELSE 0 END) AS n_iso,
    SUM(CASE WHEN label='mag' THEN 1 ELSE 0 END) AS n_mag
  FROM labeled GROUP BY family
)
SELECT
  COUNT(*) AS n_families,
  SUM(CASE WHEN n_iso>=5 AND n_mag>=5 THEN 1 ELSE 0 END) AS both_5plus,
  SUM(CASE WHEN n_iso>=10 AND n_mag>=10 THEN 1 ELSE 0 END) AS both_10plus,
  SUM(CASE WHEN n_iso>=20 AND n_mag>=20 THEN 1 ELSE 0 END) AS both_20plus
FROM fam
''').show()

## Q4 — Does the cultivability signal survive a CheckM control?

This is the key feasibility question. MAGs are by construction more incomplete than isolates, which would *artifactually* depress their pathway completeness. If we restrict to MAGs with `checkm_completeness ≥ 95%` (so the genome itself is nearly complete), does the cultivability signal in pathway completeness survive?

Two-table join with `REGEXP_REPLACE` to strip GTDB prefixes (`RS_` / `GB_`) from `gtdb_metadata.accession` to match `gapmind_pathways.genome_id`.

**Result (CheckM ≥ 95%, contam ≤ 5%):**

| metabolic_category | label | n | mean frac complete | sd | mean checkm |
|---|---|---:|---:|---:|---:|
| aa | isolate | 208,073 | 0.902 | 0.172 | 99.6 |
| aa | mag | 27,598 | **0.813** | 0.219 | 97.8 |
| carbon | isolate | 190,011 | 0.679 | 0.206 | 99.5 |
| carbon | mag | 27,581 | **0.400** | 0.192 | 97.8 |

**Signal survives**:

- **9pp** isolate-vs-MAG gap on amino-acid biosynthesis (modest)
- **28pp** isolate-vs-MAG gap on carbon utilization (large)

Mean CheckM completeness is only 1.8pp different between cohorts (99.6 vs 97.8), so the pathway-completeness gap is *not* explained by assembly incompleteness. The hypothesis has substantial signal.

In [ ]:
spark.sql('''
WITH meta AS (
  SELECT
    REGEXP_REPLACE(accession, '^(RS_|GB_)', '') AS genome_id,
    CASE WHEN ncbi_genome_category = 'none' THEN 'isolate'
         WHEN ncbi_genome_category IN ('derived from metagenome', 'derived from environmental sample', 'derived from single cell') THEN 'mag'
         ELSE 'unknown' END AS label,
    CAST(checkm_completeness AS DOUBLE) AS checkm_comp,
    CAST(checkm_contamination AS DOUBLE) AS checkm_contam
  FROM kbase_ke_pangenome.gtdb_metadata
),
gp AS (
  SELECT genome_id, metabolic_category, pathway,
         MAX(CAST(score_simplified AS DOUBLE)) AS complete
  FROM kbase_ke_pangenome.gapmind_pathways
  WHERE sequence_scope = 'all'
  GROUP BY genome_id, metabolic_category, pathway
),
per_genome AS (
  SELECT genome_id, metabolic_category,
         AVG(complete) AS frac_complete
  FROM gp GROUP BY genome_id, metabolic_category
)
SELECT metabolic_category, label,
       COUNT(*) AS n_genomes,
       ROUND(AVG(frac_complete), 3) AS mean_frac,
       ROUND(STDDEV(frac_complete), 3) AS sd_frac,
       ROUND(AVG(checkm_comp), 1) AS mean_checkm
FROM per_genome pg
JOIN meta m ON pg.genome_id = m.genome_id
WHERE m.checkm_comp >= 95 AND m.checkm_contam <= 5
GROUP BY metabolic_category, label
ORDER BY metabolic_category, label
''').show()

## Implications for the research plan

1. **Label**: use `ncbi_genome_category` binary (`'none'` = isolate, `IN ('derived from metagenome', 'derived from environmental sample', 'derived from single cell')` = MAG-or-uncultured). Drop `unknown`.
2. **Quality gate**: require `checkm_completeness ≥ 95` *and* `checkm_contamination ≤ 5` for both labels, so the model is not learning "MAG = fragmented assembly".
3. **Feature matrix**: per genome, the binary vector `MAX(score_simplified)` across 80 pathways × `sequence_scope='all'` → 80-dim feature. Plus genome-level covariates: `genome_size`, `gc_percentage`, `contig_count`, `n50_contigs`.
4. **Holdout**: family-blocked leave-out (203 families with ≥5 of both labels). Reserve ~20% of these families as held-out test set, never seen during training.
5. **Pre-registered confound check**: include `checkm_completeness` itself as a baseline feature and report the model's improvement over a baseline that has only checkm + taxonomy.
6. **Deliverable**: ranked candidate list of high-scoring MAGs (predicted-cultivable but currently uncultured) filtered by environmental relevance (subsurface, soil, plant-associated) for downstream experimental prioritization.

Hand off to `RESEARCH_PLAN.md` for Phase B.